# 🚀 Instalación de RASA directamente en Google Colab (sin Docker)

In [ ]:

# Primero instalamos dependencias de sistema necesarias (puede tardar unos minutos)
!pip install --upgrade pip
!pip install rasa
!pip install openai


# 🚀 Inicializamos proyecto RASA en el entorno de Colab

In [ ]:

!rasa init --no-prompt

# Movemos al directorio creado por rasa init
import os
os.chdir("")


# 🚀 Modificamos intents y domain

In [ ]:

# Sobreescribimos NLU
nlu_content = """
version: "3.1"
nlu:
- intent: saludo
  examples: |
    - hola
    - buenos días
    - qué tal

- intent: horario
  examples: |
    - ¿cuál es el horario del curso?
    - dime los horarios de clase

- intent: ubicacion
  examples: |
    - ¿dónde es el aula?
    - dime el aula del curso
"""

with open('./data/nlu.yml', 'w') as f:
    f.write(nlu_content)

# Sobreescribimos domain.yml
domain_content = """
version: "3.1"
intents:
  - saludo
  - horario
  - ubicacion

responses:
  utter_saludo:
  - text: "¡Hola! ¿En qué puedo ayudarte con el curso de IA?"
  
  utter_horario:
  - text: "El horario del curso es lunes y miércoles de 17:00 a 19:00."

  utter_ubicacion:
  - text: "El aula es la 3.1 del edificio principal."

actions:
  - action_fallback
"""

with open('./domain.yml', 'w') as f:
    f.write(domain_content)

# Sobreescribimos config.yml
config_content = """
version: "3.1"
policies:
  - name: RulePolicy
    core_fallback_threshold: 0.4
    core_fallback_action_name: "action_fallback"
"""

with open('./config.yml', 'w') as f:
    f.write(config_content)


# 🚀 Creamos acción personalizada LLM

In [ ]:

actions_code = """
from rasa_sdk import Action
from rasa_sdk.events import EventType
import openai

class ActionFallbackLLM(Action):
    def name(self) -> str:
        return "action_fallback"

    def run(self, dispatcher, tracker, domain) -> list[EventType]:
        user_message = tracker.latest_message.get('text')

        openai.api_key = "TU_API_KEY_AQUI"

        response = openai.Completion.create(
            engine="text-davinci-003",
            prompt=f"Responde como un asistente de un curso de IA. Pregunta: {user_message}",
            max_tokens=100
        )
        answer = response.choices[0].text.strip()

        dispatcher.utter_message(text=answer)
        return []
"""

os.makedirs('./actions', exist_ok=True)
with open('./actions/actions.py', 'w') as f:
    f.write(actions_code)

# Creamos endpoints.yml
endpoints_content = """
action_endpoint:
  url: "http://localhost:5055/webhook"
"""

with open('./endpoints.yml', 'w') as f:
    f.write(endpoints_content)

# Creamos reglas fallback
rules_content = """
version: "3.1"
rules:
- rule: fallback to LLM
  steps:
  - intent: nlu_fallback
  - action: action_fallback
"""

with open('./data/rules.yml', 'w') as f:
    f.write(rules_content)


# 🚀 Entrenamos el modelo

In [ ]:

!rasa train


# 🚀 Lanzamos RASA para pruebas (modo shell)

In [ ]:

!rasa shell
